In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from ultralytics import YOLO
from PIL import Image
import io

# ==========================================
# 0. 全局配置 (必须保留，模型结构依赖这些参数)
# ==========================================
IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4

# ==========================================
# 1. ViT 模型定义 (Split-ViT + HE适配)
# ==========================================
class DynamicAct(nn.Module):
    def __init__(self):
        super().__init__()
        # 固定系数 (HE 友好)
        self.register_buffer('a', torch.tensor(0.17))
        self.register_buffer('b', torch.tensor(0.5))
        self.register_buffer('c', torch.tensor(0.12))
    def forward(self, x): return self.a * (x**2) + self.b * x + self.c

class PolyBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 2), DynamicAct(), nn.Linear(dim * 2, dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        # Client 持有前 2 层
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return x

class ServerModel(nn.Module):
    def __init__(self, num_classes=6): # 注意这里默认改为微调后的 6 类
        super().__init__()
        # Server 持有后 2 层 + 分类头
        self.blocks = nn.ModuleList([PolyBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))


# ==========================================
# 2. 系统节点定义 (云端 & 边缘端)
# ==========================================
class CloudNode:
    def __init__(self, server_weight_path):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.server_model = ServerModel(num_classes=6).to(self.device)
        self.server_model.load_state_dict(torch.load(server_weight_path, map_location=self.device))
        self.server_model.eval()
        print("☁️ [Cloud] ServerModel 专家系统已启动...")

    def process_request(self, z_payload_bytes):
        """接收边缘端传来的特征 Z 进行推理"""
        # 反序列化接收到的特征
        buffer = io.BytesIO(z_payload_bytes)
        z = torch.load(buffer, map_location=self.device)
        
        with torch.no_grad():
            logits = self.server_model(z)
            pred_class = logits.argmax(1).item()
            confidence = torch.softmax(logits, dim=1)[0][pred_class].item()
            
        return pred_class, confidence


class EdgeNode:
    def __init__(self, yolo_weight_path, client_weight_path, cloud_api, conf_thresh=0.7):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.conf_thresh = conf_thresh
        self.cloud_api = cloud_api  # 模拟云端通信接口
        
        # 加载 YOLO 初筛模型
        self.yolo = YOLO(yolo_weight_path)
        
        # 加载 ViT 边缘特征提取模型
        self.client_model = ClientModel().to(self.device)
        self.client_model.load_state_dict(torch.load(client_weight_path, map_location=self.device))
        self.client_model.eval()
        
        # ViT 预处理
        self.transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        print(f"🏭 [Edge] YOLO & ClientModel 已启动，置信度阈值设为: {self.conf_thresh}")

    def add_privacy_noise(self, z, noise_std=1e-3):
        """推理阶段的隐私保护：向传输的特征中注入噪声"""
        noise = torch.randn_like(z) * noise_std
        return z + noise

    def infer(self, image_path):
        print(f"\n📸 处理图片: {image_path}")
        original_img = Image.open(image_path).convert('RGB')
        
        # 1. YOLO 边缘初检
        results = self.yolo(image_path, verbose=False, conf=0.05)[0]
        final_detections = []
        
        for box in results.boxes:
            conf = box.conf.item()
            cls_idx = int(box.cls.item())
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            # 2. 判断是否需要呼叫云端
            if conf >= self.conf_thresh:
                print(f"   ✅ [Edge YOLO] 置信度充足 ({conf:.2f}) -> 类别: {cls_idx}")
                final_detections.append({'box': (x1,y1,x2,y2), 'class': cls_idx, 'source': 'YOLO'})
            else:
                print(f"   ⚠️ [Edge YOLO] 置信度低 ({conf:.2f}) -> 触发云端复核！裁剪缺陷区域...")
                
                # 3. 裁剪并缩放缺陷区域
                crop_img = original_img.crop((x1, y1, x2, y2))
                input_tensor = self.transform(crop_img).unsqueeze(0).to(self.device)
                
                # 4. 边缘端提取特征
                with torch.no_grad():
                    z = self.client_model(input_tensor)
                    # 注入隐私噪声，防止云端反推原图
                    z_safe = self.add_privacy_noise(z)
                
                # 5. 序列化并发送给云端 (模拟网络传输)
                buffer = io.BytesIO()
                torch.save(z_safe.cpu(), buffer)
                z_bytes = buffer.getvalue()
                
                cloud_cls, cloud_conf = self.cloud_api.process_request(z_bytes)
                print(f"   ☁️ [Cloud ViT] 专家复核完成 ({cloud_conf:.2f}) -> 修正类别: {cloud_cls}")
                
                final_detections.append({'box': (x1,y1,x2,y2), 'class': cloud_cls, 'source': 'Cloud-ViT'})
                
        return final_detections

# ==========================================
# 3. 运行测试
# ==========================================
if __name__ == '__main__':
    # 路径配置
    YOLO_WEIGHTS = '/root/autodl-tmp/exp1/result_exp1/18_Rescue_Crazing/weights/best.pt'
    CLIENT_WEIGHTS = '/root/autodl-tmp/exp2/Fed_ViT_Models/fed_vit_client_best_v50.pth'
    SERVER_WEIGHTS = '/root/autodl-tmp/exp2/Fed_ViT_Models/fed_vit_server_best_v50.pth'
    TEST_IMAGE = '/root/autodl-tmp/exp2/data/NEU/IMAGES/crazing_1.jpg' 
    
    # 实例化节点 (现在可以正常找到 ServerModel 等定义了)
    cloud_node = CloudNode(SERVER_WEIGHTS)
    edge_node = EdgeNode(YOLO_WEIGHTS, CLIENT_WEIGHTS, cloud_api=cloud_node, conf_thresh=0.7)
    
    # 执行流水线
    results = edge_node.infer(TEST_IMAGE)
    
    print("\n📊 最终检测结果汇总:")
    for res in results:
        print(f"   - BBox: {res['box']}, Class: {res['class']}, Provided by: {res['source']}")

☁️ [Cloud] ServerModel 专家系统已启动...
🏭 [Edge] YOLO & ClientModel 已启动，置信度阈值设为: 0.7

📸 处理图片: /root/autodl-tmp/exp2/data/NEU/IMAGES/crazing_1.jpg
   ⚠️ [Edge YOLO] 置信度低 (0.14) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.50) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.12) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.66) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.12) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.70) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.11) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.59) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.10) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.67) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.08) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.49) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.08) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.50) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.08) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.71) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 (0.07) -> 触发云端复核！裁剪缺陷区域...
   ☁️ [Cloud ViT] 专家复核完成 (0.58) -> 修正类别: 0
   ⚠️ [Edge YOLO] 置信度低 